In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case)

C:\Users\Lena\Promotion\neurolib\GUI\current\gui\data\00120
00120


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

In [7]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [8]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [9]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1':
    cntrl_vars_init = [1]
elif case[3] == '2':
    cntrl_vars_init = [0,1]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [10]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_' + case + '.pickle'
case_1 = case[0] + case[1] + '0' + case[3] + case[4]
final_file_1 = 'control_' + case_1 + '.pickle'

In [11]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [12]:
# get initial parameters and target states

i_stepsize = 4
limit = 40
i_range = range(0, limit,i_stepsize)
i_range_0 = range(0, limit,i_stepsize)
i_range_1 = range(0, limit,i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  4 0.4500000000000001 0.3750000000000001
-------  8 0.47500000000000014 0.40000000000000013
-------  12 0.47500000000000014 0.42500000000000016
-------  16 0.47500000000000014 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  24 0.4000000000000001 0.5000000000000002
-------  28 0.5000000000000002 0.5000000000000002
-------  32 0.47500000000000014 0.5250000000000002
-------  36 0.4250000000000001 0.5500000000000003


In [13]:
# get uncontrolled cost

data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  4 0.4500000000000001 0.3750000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13602.2666253313
Gradient descend method:  None
RUN  0 , total integrated cost =  13602.2666253313
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  8 0.47500000000000014 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17551.147823015366
Gradient descend method:  None
RUN  0 , total integrated cost =  17551.147823015366
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  12 0.47500000000000014 0.4250000000000001

In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    
    if not type(bestControl_init[i]) == type(None):
        continue
        
    control0 = aln.getZeroControl()

    ##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)

    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    with open(init_file,'wb') as f:
        pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

-------  0 0.4000000000000001 0.3500000000000001
-------  4 0.4500000000000001 0.3750000000000001
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13602.2666253313
Gradient descend method:  None
RUN  1 , total integrated cost =  147.34724798663964
RUN  2 , total integrated cost =  24.216457780062868
RUN  3 , total integrated cost =  22.372701250313252
RUN  4 , total integrated cost =  22.22152630943582
RUN  5 , total integrated cost =  21.955869019259566
RUN  6 , total integrated cost =  21.846571201834553
RUN  7 , total integrated cost =  21.615958584029404
RUN  8 , total integrated cost =  21.51535772701688
RUN  9 , total integrated cost =  21.30020065066863
RUN  10 , total integrated cost =  21.18765857080294
RUN  11 , total integrated cost =  16.76646313996479
RUN  12 , total integrated cost =  16.25589543991747
RUN  13 , total integrated cost =  16.057311824419827
RUN  14 , total integrated cost =  15.764953551828764
RUN  15 , 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  66 , total integrated cost =  5.804158251825341
Improved over  66  iterations in  49.232365300000005  seconds by  99.95732947741948  percent.
Problem in initial value trasfer:  Vmean_exc -56.67605480582644 -56.6760549839155
weight =  23435.382074659225
set cost params:  1.0 0.0 23435.382074659225
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13542.981862676923
Gradient descend method:  None
RUN  1 , total integrated cost =  12856.97576253328
RUN  2 , total integrated cost =  12844.139580101315
RUN  3 , total integrated cost =  12839.69672512058
RUN  4 , total integrated cost =  12839.696725120553
RUN  5 , total integrated cost =  12839.696725120548
RUN  6 , total integrated cost =  12839.696725120539
RUN  7 , total integrated cost =  12839.696725120537
State only changes marginally.


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  12839.696725120537
Control only changes marginally.
RUN  8 , total integrated cost =  12839.696725120537
Improved over  8  iterations in  2.7548844999999886  seconds by  5.192985892527616  percent.
Problem in initial value trasfer:  Vmean_exc -56.67585621401076 -56.675861528639025
-------  8 0.47500000000000014 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17551.147823015366
Gradient descend method:  None
RUN  1 , total integrated cost =  39.56439472650375
RUN  2 , total integrated cost =  25.84911102676776
RUN  3 , total integrated cost =  16.92337849947612
RUN  4 , total integrated cost =  14.245156098243028
RUN  5 , total integrated cost =  13.742802988688819
RUN  6 , total integrated cost =  13.260147865693398
RUN  7 , total integrated cost =  12.844964149873034
RUN  8 , total integrated cost =  12.138251776392778
RUN  9 , total integrated cost =  11.561597192327637
RUN  1

...

Control only changes marginally.
RUN  163 , total integrated cost =  38.683245724844475
Improved over  163  iterations in  21.2743217  seconds by  99.81850008292751  percent.
Problem in initial value trasfer:  Vmean_exc -56.697857017794746 -56.697857207399764
weight =  5509.644390639601
set cost params:  1.0 0.0 5509.644390639601
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21289.794586741988
Gradient descend method:  None
RUN  1 , total integrated cost =  19577.296614485447
RUN  2 , total integrated cost =  18610.652220795102
RUN  3 , total integrated cost =  18016.27156524285
RUN  4 , total integrated cost =  17918.52955678728
RUN  5 , total integrated cost =  17914.633450189292
RUN  6 , total integrated cost =  17902.39430239089
RUN  7 , total integrated cost =  17826.132151324116
RUN  8 , total integrated cost =  17807.63546755011
RUN  9 , total integrated cost =  17805.05845578466
RUN  10 , total integrated cost =  17792.923289372007
RUN  11 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  62 , total integrated cost =  16170.037644090447
Improved over  62  iterations in  12.8106918  seconds by  24.04793959750002  percent.
Problem in initial value trasfer:  Vmean_exc -56.69781164346604 -56.697813127749235
-------  32 0.47500000000000014 0.5250000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  16568.218379642258
Gradient descend method:  None
RUN  1 , total integrated cost =  3953.9768172758113
RUN  2 , total integrated cost =  543.1205482222656
RUN  3 , total integrated cost =  458.7241957265865
RUN  4 , total integrated cost =  386.75915419687306
RUN  5 , total integrated cost =  331.54219953411115
RUN  6 , total integrated cost =  271.31405701125556
RUN  7 , total integrated cost =  230.63065306159945
RUN  8 , total integrated cost =  193.28285133219035
RUN  9 , total integrated cost =  168.7916656406834
RUN  10 , total integrated cost =  137.90832765818053
RUN  11 , t

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  33 , total integrated cost =  15413.970047948484
Improved over  33  iterations in  8.98419530000001  seconds by  6.932812607219603  percent.
Problem in initial value trasfer:  Vmean_exc -56.685638214707595 -56.685641344266635
-------  36 0.4250000000000001 0.5500000000000003
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7741.667733280556
Gradient descend method:  None
RUN  1 , total integrated cost =  7741.667733280556
Control only changes marginally.
RUN  1 , total integrated cost =  7741.667733280556
Improved over  1  iterations in  0.491775500000017  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7741.667733280556
Gradient descend method:  None
RUN  1 , total integrated cost =  7741.667733280556
Control only changes marginally.
RUN  1 , total integrated cost =  7741.667733280556
Improved over  

In [15]:
"""
#plot initial guesses
for i in i_range:
    print("---------", i)
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
    plt.show()
"""

'\n#plot initial guesses\nfor i in i_range:\n    print("---------", i)\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n    plt.show()\n'

In [17]:
found_solution = []
no_solution = []
factor_iteration = 1.
already_tried = [ [] for _ in range(len(exc)) ]

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break


    for i in i_range:
        print("------- ", i, exc[i], inh[i])        

        if np.abs(np.mean(bestState_init[i][0,0,-300:]) - target[i][0,0,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,0,-100:]) - bestState_init[i][0,0,0]) and np.abs(
            np.mean(bestState_init[i][0,1,-300:]) - target[i][0,1,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,1,-100:]) - bestState_init[i][0,1,0]) and np.amax(
            bestState_init[i][0,0,:]) < target[i][0,0,-1] + 1. and np.amax(
            bestState_init[i][0,1,:]) < target[i][0,1,-1] + 1.:
            # and np.amin(bestState_init[i][0,0,:]) > bestState_init[i][0,0,0] - 1.
            #and np.amin(bestState_init[i][0,1,:]) > bestState_init[i][0,1,0] - 1.:
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        if len(found_solution) == 0:
            continue
            
        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)

------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  4 0.4500000000000001 0.3750000000000001
found solution for  4
-------  8 0.47500000000000014 0.40000000000000013
found solution for  8
-------  12 0.47500000000000014 0.42500000000000016
found solution for  12
-------  16 0.47500000000000014 0.4500000000000002
found solution for  16
-------  20 0.4500000000000001 0.4750000000000002
found solution for  20
-------  24 0.4000000000000001 0.5000000000000002
[0, 4, 8, 12, 16, 20] []
closest index  20
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3017.025741174337
Gradient descend method:  None
RUN  1 , total integrated cost =  329.22908213708547
RUN  2 , total integrated cost =  255.8114374735349
RUN  3 , 

ERROR:root:Problem in initial value trasfer


RUN  100 , total integrated cost =  72.38754222288613
RUN  100 , total integrated cost =  72.38754222288613
Improved over  100  iterations in  14.192062400000054  seconds by  97.6006985543746  percent.
Problem in initial value trasfer:  Vmean_exc -56.65183910500674 -56.651840308481226
weight =  411.80865392762564
set cost params:  1.0 0.0 411.80865392762564
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2963.750017085236
Gradient descend method:  None
RUN  1 , total integrated cost =  2951.7677729003444
RUN  2 , total integrated cost =  2941.3104885021216
RUN  3 , total integrated cost =  2932.329106041818
RUN  4 , total integrated cost =  2925.401879208466
RUN  5 , total integrated cost =  2918.0426795043486
RUN  6 , total integrated cost =  2912.1654237136886
RUN  7 , total integrated cost =  2906.332825326233
RUN  8 , total integrated cost =  2901.5990254625767
RUN  9 , total integrated cost =  2896.7729148150643
RUN  10 , total integrated cost =  2892.55566

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  85 , total integrated cost =  2688.98711485669
Improved over  85  iterations in  19.9783013  seconds by  9.27078534439849  percent.
Problem in initial value trasfer:  Vmean_exc -56.64790489102963 -56.647942691856954
-------  28 0.5000000000000002 0.5000000000000002
found solution for  28
-------  32 0.47500000000000014 0.5250000000000002
found solution for  32
-------  36 0.4250000000000001 0.5500000000000003
[0, 4, 8, 12, 16, 20, 28, 32] []
closest index  32
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7782.762302090978
Gradient descend method:  None
RUN  1 , total integrated cost =  7742.631412025367
RUN  2 , total integrated cost =  7741.680600516179
RUN  3 , total integrated cost =  7741.6683933568975
RUN  4 , total integrated cost =  7741.667740783179
RUN  5 , total integrated cost =  7741.667733332672
RUN  6 , total integrated cost =  7741.667733280827
RUN  7 , to

ERROR:root:Problem in initial value trasfer


RUN  500 , total integrated cost =  69.17027220836829
RUN  500 , total integrated cost =  69.17027220836829
Improved over  500  iterations in  114.74464119999993  seconds by  99.10651975011776  percent.
Problem in initial value trasfer:  Vmean_exc -56.636127611066456 -56.636127547920275
------------------------------------------------------------
-------------------- 1
------------------------------------------------------------
found solution:  [0, 4, 8, 12, 16, 20, 28, 32]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  4 0.4500000000000001 0.3750000000000001
-------  8 0.47500000000000014 0.40000000000000013
-------  12 0.47500000000000014 0.42500000000000016
-------  16 0.47500000000000014 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  24 0.4000000000000001 0.5000000000000002
found solution for  24
-------  28 0.5000000000000002 0.5000000000000002
-------  32 0.47500000000000014 0.5250000000000002
-------  36 0.4250000000000

In [18]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print("------------------------------------------------")
    print('-------------------------', counter)
    
    if counter > 20:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_init[i] == [True, True]:
            continue
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
                       
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
    counter += 1

------------------------------------------------
------------------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  5835.454352552801
set cost params:  1.0 0.0 5835.454352552801
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5901.3951792054
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5901.3951792054
Control only changes marginally.
RUN  1 , total integrated cost =  5901.3951792054
Improved over  1  iterations in  0.8410407999999734  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.6262228967628 -56.62623400496918
converged for  0
-------  4 0.4500000000000001 0.3750000000000001
weight =  24826.246489580306
set cost params:  1.0 0.0 24826.246489580306
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13596.242127341708
Gradient descend method:  None
RUN  1 , total integrated cost =  13596.125621713281
RUN  2 , total integrated cost =  13596.124955006891
RUN  3 , total integrated cost =  13596.124955006839
RUN  4 , total integrated cost =  13596.124955006826
RUN  5 , total integrated cost =  13596.124955006819
State only changes marginally.


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  13596.124955006819
Control only changes marginally.
RUN  6 , total integrated cost =  13596.124955006819
Improved over  6  iterations in  3.884921400000053  seconds by  0.0008617994133288676  percent.
Problem in initial value trasfer:  Vmean_exc -56.67584965886342 -56.67585514708964
no convergence
-------  8 0.47500000000000014 0.40000000000000013
weight =  37483.91590254708
set cost params:  1.0 0.0 37483.91590254708
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17533.461083517617
Gradient descend method:  None
RUN  1 , total integrated cost =  17532.881051348668
RUN  2 , total integrated cost =  17532.877711683082
RUN  3 , total integrated cost =  17532.877652139356
RUN  4 , total integrated cost =  17532.877651087754
RUN  5 , total integrated cost =  17532.877651064297
RUN  6 , total integrated cost =  17532.87765106406
RUN  7 , total integrated cost =  17532.877651064045
RUN  8 , total integrated cost =  17532.877651063965

...

RUN  1 , total integrated cost =  21310.171221149274
Control only changes marginally.
RUN  1 , total integrated cost =  21310.171221149274
Improved over  1  iterations in  0.6496578  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.697802287500494 -56.697804094588946
converged for  28
-------  32 0.47500000000000014 0.5250000000000002
weight =  4002.7010472296797
set cost params:  1.0 0.0 4002.7010472296797
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  16564.08015398943
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  16564.08015398943
Control only changes marginally.
RUN  1 , total integrated cost =  16564.08015398943
Improved over  1  iterations in  0.6125084000000243  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.68563125475033 -56.68563456977638
converged for  32
-------  36 0.4250000000000001 0.5500000000000003
------------------------------------------------
------------------------- 6
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
-------  0 0.4000000000000001 0.3500

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  21310.171221281063
Control only changes marginally.
RUN  1 , total integrated cost =  21310.171221281063
Improved over  1  iterations in  0.6198659000000362  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.697802287500494 -56.697804094588946
converged for  28
-------  32 0.47500000000000014 0.5250000000000002
-------  36 0.4250000000000001 0.5500000000000003
------------------------------------------------
------------------------- 7
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, Tru

In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
factor_iteration = 20
conv_0 = [[False]*2] * len(exc)
full_converge = False

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 20:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break
        
    counter += 1
    
    for i in i_range_0:
        print("------- ", i, exc[i], inh[i])
        
        if conv_0[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

    # exc and inh control current 

        setinit(initVars[i], aln)
        aln.params.duration = dur

        if not type(bestControl_0[i]) == type(None):
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
        else:
            control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
            weights_0[i] = weights_init[i]
            cost_0[i] = cost_init[i]

        cgv = None
        max_it = 500 * factor_iteration

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                           + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        with open(final_file,'wb') as f:
            pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                     costnode_0, weights_0], f)
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  5835.454352552801
set cost params:  1.0 0.0 5835.454352552801
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5901.3951792054
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5901.3951792054
Control only changes marginally.
RUN  1 , total integrated cost =  5901.3951792054
Improved over  1  iterations in  0.5746493000000328  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.6262228967628 -56.62623400496918
converged for  0
-------  4 0.4500000000000001 0.3750000000000001
weight =  24836.534407251438
set cost params:  1.0 0.0 24836.534407251438
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13601.718975697288
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  13601.718975697288
Control only changes marginally.
RUN  1 , total integrated cost =  13601.718975697288
Improved over  1  iterations in  0.6000887000000148  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67584963929095 -56.67585512803304
converged for  4
-------  8 0.47500000000000014 0.40000000000000013
weight =  37522.48249437282
set cost params:  1.0 0.0 37522.48249437282
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17550.680085297063
Gradient descend method:  None


...

RUN  1 , total integrated cost =  21310.17122128334
Control only changes marginally.
RUN  1 , total integrated cost =  21310.17122128334
Improved over  1  iterations in  0.5443657999999232  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.697802287500494 -56.697804094588946
converged for  28
-------  32 0.47500000000000014 0.5250000000000002
weight =  4002.7010472296947
set cost params:  1.0 0.0 4002.7010472296947
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  16564.080153989493
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  16564.080153989493
Control only changes marginally.
RUN  1 , total integrated cost =  16564.080153989493
Improved over  1  iterations in  0.6470520000000306  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.68563125475033 -56.68563456977638
converged for  32
-------  36 0.4250000000000001 0.5500000000000003
weight =  1169.2924065670484
set cost params:  1.0 0.0 1169.2924065670484
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7735.052576513034
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  7735.052576513034
Control only changes marginally.
RUN  1 , total integrated cost =  7735.052576513034
Improved over  1  iterations in  0.8830397000000403  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.63632530048785 -56.63632139208764
converged for  36
--------------- 2
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
full convergence


In [21]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [22]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [23]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    if counter > 20:
        break
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_1[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        if not type(bestControl_1[i]) == type(None):
            control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1].copy()
        else:
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1].copy()
            cost_1[i] = cost_0[i]
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(final_file_1,'wb') as f:
            pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)
            
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
    counter += 1

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.1130619408485818
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.1130619408485818
Control only changes marginally.
RUN  1 , total integrated cost =  1.1130619408485818
Improved over  1  iterations in  0.6124182999999448  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.627622079911795 -56.62762204082369
converged for  0
-------  4 0.4500000000000001 0.3750000000000001
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  97.49188710198413
Gradient descend method:  None
RUN  1 , total integrated cost =  0.5975356789505769
RUN  2 , total integrated cost =  0.5934032170406949
RUN  3 , total integrated cost =  0.5919472640056795
RUN  4 , total integrated cost =  0.5907440866479651
RUN  5 , total integrated cost =  0.5900209095169833
RUN  6 , total integrated cost =  0.5894580803370519
RUN  7 , total integrated cost =  0.58913268714952
RUN  8 , total integrated cost =  0.588808489439224
RUN  9 , total integrated cost =  0.5886701011803985
RUN  10

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  142 , total integrated cost =  0.5840960208663505
Improved over  142  iterations in  33.21821679999994  seconds by  99.40087730555945  percent.
Problem in initial value trasfer:  Vmean_exc -56.67605687511343 -56.67605688554732
no convergence
-------  8 0.47500000000000014 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  231.09053351522567
Gradient descend method:  None
RUN  1 , total integrated cost =  0.5281578219981623
RUN  2 , total integrated cost =  0.5246355092033266
RUN  3 , total integrated cost =  0.5209984390581861
RUN  4 , total integrated cost =  0.5193270594825096
RUN  5 , total integrated cost =  0.5174108774841578
RUN  6 , total integrated cost =  0.5162000461830255
RUN  7 , total integrated cost =  0.5138922931245453
RUN  8 , total integrated cost =  0.5117439541711717
RUN  9 , total integrated cost =  0.5094687153397006
RUN  10 , total integrated cost =  0.508

...

RUN  1 , total integrated cost =  6.538237043140779
Control only changes marginally.
RUN  1 , total integrated cost =  6.538237043140779
Improved over  1  iterations in  0.67101619999994  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.65188295386897 -56.65188294042322
converged for  24
-------  28 0.5000000000000002 0.5000000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3.067679743834042
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  3.067679743834042
Control only changes marginally.
RUN  1 , total integrated cost =  3.067679743834042
Improved over  1  iterations in  0.6080647999997382  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69785465082642 -56.69785468608287
converged for  28
-------  32 0.47500000000000014 0.5250000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4.152601720710101
Gradient descend method:  None
RUN  1 , total integrated cost =  4.152601720710101
Control only changes marginally.
RUN  1 , total integrated cost =  4.152601720710101
Improved over  1  iterations in  0.8326622999998108  seconds by  0.0  percent.
converged for  32
-------  36 0.4250000000000001 0.5500000000000003
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6.631981683771573
Gradient descend method:  None
RUN  1 , total integrated cost =  6.6319816837

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  3.067679743834042
Control only changes marginally.
RUN  1 , total integrated cost =  3.067679743834042
Improved over  1  iterations in  0.6475368999999773  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69785465082642 -56.69785468608287
converged for  28
-------  32 0.47500000000000014 0.5250000000000002
-------  36 0.4250000000000001 0.5500000000000003
--------------- 4
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
full convergence
